<a href="https://colab.research.google.com/github/utsavmakavana455-ops/Berlin-Housing-Price-Analysis/blob/main/college_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**# Berlin Housing Price Analysis**
### **Loads the dataset, cleans it, treats outliers, engineers a few features,**
# then looks at what drives price (sqft_total and living area).

In [3]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set visual style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100


# CONFIGURATION

DATA_PATH = "/content/BerlinHousing4049 (1).csv"      # UPDATE THIS PATH as needed
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# 1. DATA LOADING AND INITIAL INSPECTION

df = None
for enc in ["utf-8", "latin1", "cp1252"]:
    for sep in [",", ";", "\t"]:
        try:
            candidate = pd.read_csv(DATA_PATH, encoding=enc, sep=sep)
            if candidate.shape[1] > 1:
                df = candidate
                print(f"Successfully loaded with encoding='{enc}', sep='{sep}'")
                break
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue
    if df is not None:
        break
if df is None:
    df = pd.read_csv(DATA_PATH, sep=None, engine="python", encoding="latin1")
    print("Loaded with inferred separator (engine='python')")

print("=" * 60)
print("RAW DATA OVERVIEW")
print("=" * 60)
print(f"Shape: {df.shape}")
print(df.info())
print(f"\nDuplicated rows: {df.duplicated().sum()}")
print("\nMissing values per column:")
print(df.isnull().sum())
print(f"\nSkewness (raw): price = {df['price'].skew():.2f}, "
      f"sqft_total = {df['sqft_total'].skew():.2f}")


# 2. DATA CLEANING

df_clean = df.drop_duplicates().copy()
df_clean = df_clean.dropna(subset=["price"])

for col in ["bedrooms", "bathrooms", "sqft_total", "built"]:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f"Imputed {col} with median: {median_val}")

mode_condition = df_clean["condition"].mode()[0]
df_clean["condition"] = df_clean["condition"].fillna(mode_condition)
print(f"Imputed condition with mode: {mode_condition}")

df_clean["living_area"] = df_clean["living_area_sqft"].fillna(df_clean["sqft_living"])
living_median = df_clean["living_area"].median()
df_clean["living_area"] = df_clean["living_area"].fillna(living_median)
print(f"Unified living_area created; final median: {living_median}")
print(f"Rows in cleaned dataset: {len(df_clean)}")


# 3. OUTLIER TREATMENT - TUKEY IQR WINSORISATION (CAPPING)

def iqr_fences(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1, q3, iqr, q1 - 1.5 * iqr, q3 + 1.5 * iqr

for original, capped in [("price", "price_capped"),
                         ("sqft_total", "sqft_capped"),
                         ("living_area", "living_capped")]:
    q1, q3, iqr, lo, hi = iqr_fences(df_clean[original])
    n_out = ((df_clean[original] < lo) | (df_clean[original] > hi)).sum()
    print(f"{original}: fences [{lo:,.0f}, {hi:,.0f}], "
          f"outliers capped = {n_out} ({n_out / len(df_clean) * 100:.1f}%)")
    df_clean[capped] = df_clean[original].clip(lo, hi)

print(f"\nSkewness after capping: price = {df_clean['price_capped'].skew():.2f}, "
      f"sqft_total = {df_clean['sqft_capped'].skew():.2f}")

# 4. FEATURE ENGINEERING

df_clean["price_per_sqft_living"] = df_clean["price_capped"] / df_clean["living_capped"]
df_clean["property_age"] = 2026 - df_clean["built"]
df_clean["is_renovated"] = (df_clean["renovated"] > 0).astype(int)
df_clean["total_rooms"] = df_clean["bedrooms"] + df_clean["bathrooms"]
df_clean["grade_category"] = pd.cut(
    df_clean["grade"], bins=[-np.inf, 5, 7, 9, np.inf],
    labels=["Low (<=5)", "Medium (6-7)", "High (8-9)", "Luxury (>=10)"])
df_clean["age_group"] = pd.cut(
    df_clean["built"], bins=[-np.inf, 1950, 1980, 2000, np.inf],
    labels=["Before 1950", "1950-1980", "1981-2000", "After 2000"])
print(f"Feature engineering complete. Final shape: {df_clean.shape}")


# 5. HELPER: 95% CONFIDENCE INTERVAL FOR PEARSON

def pearson_ci(r, n, alpha=0.05):
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    zc = stats.norm.ppf(1 - alpha / 2)
    return np.tanh(z - zc * se), np.tanh(z + zc * se)


# 6. VISUALIZATION 1: BEFORE VS AFTER OUTLIER TREATMENT

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.boxplot(x=df["price"], ax=axes[0, 0], color="lightcoral")
axes[0, 0].set_title("Price - Before Capping", fontsize=11)
axes[0, 0].set_xlabel("Price (£)")

sns.boxplot(x=df_clean["price_capped"], ax=axes[0, 1], color="lightgreen")
axes[0, 1].set_title("Price - After Capping", fontsize=11)
axes[0, 1].set_xlabel("Price (£)")

sns.histplot(df["sqft_total"].dropna(), bins=40, ax=axes[1, 0],
             color="lightcoral", kde=True)
axes[1, 0].set_title("Sqft Total - Before Capping", fontsize=11)
axes[1, 0].set_xlabel("Total Square Feet")

sns.histplot(df_clean["sqft_capped"], bins=40, ax=axes[1, 1],
             color="lightgreen", kde=True)
axes[1, 1].set_title("Sqft Total - After Capping", fontsize=11)
axes[1, 1].set_xlabel("Total Square Feet")

plt.suptitle("Figure 1 - Impact of Winsorization on Price and Sqft Total",
             fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure1_outliers.png"), dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved: figure1_outliers.png")


# 7. INVESTIGATION 1: PRICE vs TOTAL LOT SIZE

r1, p1 = stats.pearsonr(df_clean["price_capped"], df_clean["sqft_capped"])
s1, sp1 = stats.spearmanr(df_clean["price_capped"], df_clean["sqft_capped"])
lo1, hi1 = pearson_ci(r1, len(df_clean))
print(f"\nInvestigation 1 - price vs lot size: Pearson r = {r1:.3f} "
      f"[95% CI {lo1:.3f}, {hi1:.3f}], p = {p1:.2e}; Spearman rho = {s1:.3f}")
print(f"Coefficient of determination R^2 = {r1**2 * 100:.1f}%")

fig, ax = plt.subplots(figsize=(8, 6))
sns.regplot(
    x="sqft_capped", y="price_capped", data=df_clean,
    scatter_kws={"alpha": 0.4, "s": 15},
    line_kws={"color": "red", "linewidth": 2},
    ax=ax
)
ax.set_title(f"Figure 2 - Price vs Total Sqft (Pearson r = {r1:.3f})", fontsize=13, fontweight="bold")
ax.set_xlabel("Total Square Feet", fontsize=11)
ax.set_ylabel("Price (£)", fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure2_price_vs_sqft.png"), dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved: figure2_price_vs_sqft.png")


# 8. INVESTIGATION 2: PRICE vs LIVING AREA

r2, p2 = stats.pearsonr(df_clean["price_capped"], df_clean["living_capped"])
s2, sp2 = stats.spearmanr(df_clean["price_capped"], df_clean["living_capped"])
lo2, hi2 = pearson_ci(r2, len(df_clean))
reg = stats.linregress(df_clean["living_capped"], df_clean["price_capped"])
tcrit = stats.t.ppf(0.975, len(df_clean) - 2)
print(f"\nInvestigation 2 - price vs living area: Pearson r = {r2:.3f} "
      f"[95% CI {lo2:.3f}, {hi2:.3f}]; Spearman rho = {s2:.3f}")
print(f"OLS fit: price = {reg.slope:.2f} x living_area + {reg.intercept:,.0f}, "
      f"slope 95% CI [{reg.slope - tcrit * reg.stderr:.2f}, "
      f"{reg.slope + tcrit * reg.stderr:.2f}], R^2 = {reg.rvalue**2:.3f}")
print(f"Price per sq ft of living area: median = "
      f"£{df_clean['price_per_sqft_living'].median():.2f}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.regplot(
    x="living_capped", y="price_capped", data=df_clean,
    scatter_kws={"alpha": 0.4, "s": 15, "color": "darkgreen"},
    line_kws={"color": "red", "linewidth": 2},
    ax=ax
)
ax.set_title(f"Figure 3 - Price vs Living Area (Pearson r = {r2:.3f})", fontsize=13, fontweight="bold")
ax.set_xlabel("Living Area (Sqft)", fontsize=11)
ax.set_ylabel("Price (£)", fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure3_price_vs_living_area.png"), dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved: figure3_price_vs_living_area.png")


# 9. INVESTIGATION 3: COMPARING THE TWO LIVING-AREA MEASURES

sub = df_clean[["price_capped", "sqft_living", "living_area_sqft"]].dropna()
r3, _ = stats.pearsonr(sub["sqft_living"], sub["living_area_sqft"])
r3p, _ = stats.pearsonr(sub["price_capped"], sub["sqft_living"])
print(f"\nInvestigation 3: sqft_living vs living_area_sqft: r = {r3:.3f} "
      f"(shared variance = {r3**2 * 100:.0f}%)")
print(f"price vs original sqft_living: r = {r3p:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(sub["sqft_living"], sub["living_area_sqft"], alpha=0.3, s=10)
axes[0].plot([sub["sqft_living"].min(), sub["sqft_living"].max()],
             [sub["sqft_living"].min(), sub["sqft_living"].max()],
             "r--", linewidth=2, label="y=x (perfect agreement)")
axes[0].set_xlabel("sqft_living")
axes[0].set_ylabel("living_area_sqft")
axes[0].set_title(f"sqft_living vs living_area_sqft\n(r = {r3:.3f})")
axes[0].legend()

data_to_plot = [sub["sqft_living"], sub["living_area_sqft"]]
axes[1].boxplot(data_to_plot, tick_labels=["sqft_living", "living_area_sqft"])
axes[1].set_ylabel("Square Feet")
axes[1].set_title("Distribution Comparison")

plt.suptitle("Figure 4 - Comparison of Living Area Measures", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure4_living_measures.png"), dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved: figure4_living_measures.png")


# 10. INVESTIGATION 4: PRICE BY PROPERTY ATTRIBUTES

grade_stats = df_clean.groupby("grade_category", observed=True).agg(
    count=("price_capped", "size"),
    mean_price=("price_capped", "mean"),
    median_price=("price_capped", "median"),
    avg_price_per_sqft=("price_per_sqft_living", "mean"))
print("\nGrade Category Statistics:")
print(grade_stats.round(0))

age_stats = df_clean.groupby("age_group", observed=True)["price_capped"].agg(["median", "size"])
print("\nAge Group Statistics:")
print(age_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.boxplot(x="grade_category", y="price_capped", data=df_clean, ax=axes[0], palette="viridis")
axes[0].set_title("Price Distribution by Grade Category", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Grade Category")
axes[0].set_ylabel("Price (£)")
axes[0].tick_params(axis="x", rotation=15)

sns.boxplot(x="age_group", y="price_capped", data=df_clean, ax=axes[1], palette="coolwarm")
axes[1].set_title("Price Distribution by Age Group", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Age Group")
axes[1].set_ylabel("Price (£)")
axes[1].tick_params(axis="x", rotation=15)

plt.suptitle("Figure 5 - Price by Property Attributes", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure5_price_by_attributes.png"), dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved: figure5_price_by_attributes.png")

# 11. VISUALIZATION 6: CORRELATION HEATMAP

numeric_cols = ["price_capped", "sqft_capped", "living_capped", "bedrooms",
                "bathrooms", "floors", "condition", "grade", "property_age",
                "total_rooms", "price_per_sqft_living"]
corr_matrix = df_clean[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Figure 6 - Correlation Matrix of Numeric Variables", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure6_correlation_heatmap.png"), dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved: figure6_correlation_heatmap.png")


# 12. VISUALIZATION 7: PRICE PER SQFT BY GRADE

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(x="grade_category", y="price_per_sqft_living", data=df_clean,
            palette="viridis", ax=ax)
ax.set_title("Figure 7 - Price per Sqft by Grade Category", fontsize=13, fontweight="bold")
ax.set_xlabel("Grade Category")
ax.set_ylabel("Price per Sqft (£)")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure7_price_per_sqft_by_grade.png"), dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved: figure7_price_per_sqft_by_grade.png")

# 13. VISUALIZATION 8: RENOVATION IMPACT

fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(x="is_renovated", y="price_capped", data=df_clean,
            palette=["lightcoral", "lightgreen"], ax=ax)
ax.set_title("Figure 8 - Price Distribution: Renovated vs Non-Renovated",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Renovated (0 = No, 1 = Yes)")
ax.set_ylabel("Price (£)")

for i, renovated in enumerate([0, 1]):
    subset = df_clean[df_clean["is_renovated"] == renovated]["price_capped"]
    median_val = subset.median()
    ax.text(i, median_val + 50000, f"Median: £{median_val:,.0f}",
            ha="center", fontsize=10, fontweight="bold", color="darkblue")

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure8_renovation_impact.png"), dpi=200, bbox_inches="tight")
plt.close(fig)
print("Saved: figure8_renovation_impact.png")


# 14. EXPORT CLEANED DATASET

out_path = os.path.join(OUTPUT_DIR, "berlin_housing_cleaned.csv")
df_clean.to_csv(out_path, index=False)
print(f"\nCleaned dataset exported to: {out_path}")
print(f"Final dimensions - Rows: {df_clean.shape[0]}, Columns: {df_clean.shape[1]}")
print(f"\nAll 8 figures saved to: {OUTPUT_DIR}/")

Successfully loaded with encoding='utf-8', sep=','
RAW DATA OVERVIEW
Shape: (9999, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   price             9990 non-null   float64
 1   bedrooms          9988 non-null   float64
 2   bathrooms         9990 non-null   float64
 3   sqft_living       9989 non-null   float64
 4   sqft_total        9993 non-null   float64
 5   floors            9999 non-null   float64
 6   condition         9998 non-null   float64
 7   grade             9999 non-null   int64  
 8   built             9996 non-null   float64
 9   renovated         9999 non-null   int64  
 10  living_area_sqft  9986 non-null   float64
dtypes: float64(9), int64(2)
memory usage: 859.4 KB
None

Duplicated rows: 2

Missing values per column:
price                9
bedrooms            11
bathrooms            9
sqft_living         1

/tmp/ipykernel_3481/3255702407.py:247: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="grade_category", y="price_capped", data=df_clean, ax=axes[0], palette="viridis")
/tmp/ipykernel_3481/3255702407.py:253: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="age_group", y="price_capped", data=df_clean, ax=axes[1], palette="coolwarm")


Saved: figure5_price_by_attributes.png
Saved: figure6_correlation_heatmap.png


/tmp/ipykernel_3481/3255702407.py:285: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="grade_category", y="price_per_sqft_living", data=df_clean,


Saved: figure7_price_per_sqft_by_grade.png


/tmp/ipykernel_3481/3255702407.py:299: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="is_renovated", y="price_capped", data=df_clean,


Saved: figure8_renovation_impact.png

Cleaned dataset exported to: outputs/berlin_housing_cleaned.csv
Final dimensions - Rows: 9988, Columns: 21

All 8 figures saved to: outputs/
